In [2]:
# ============================================================
# NOTEBOOK 03 — VALIDATION
# Think of this like a final inspection before shipping.
# A factory checks every product before it leaves.
# We check every rule before data enters Workday.
# Each rule either PASSES ✅ or FAILS ❌
# ============================================================

# Cell 1 — Import libraries
import pandas as pd

print("✅ Libraries loaded!")

✅ Libraries loaded!


In [4]:
# Cell 2 — Load the CLEANED data
# We validate the cleaned file, not the raw one.
# This is the file that would actually go into Workday.

df = pd.read_csv("../Data/raw_hr_data.csv")

print(f"📊 Loaded cleaned data: {df.shape[0]} rows, {df.shape[1]} columns")

📊 Loaded cleaned data: 1470 rows, 33 columns


In [6]:
# Cell 3 — Set up our validation tracker
# We'll store every rule result in this list.
# At the end we print a full pass/fail report.
# Think of it like a checklist a pilot runs before takeoff.

results = []

def check(rule_name, passed, detail=""):
    # This function records one rule result.
    # rule_name = what we're checking
    # passed = True (✅) or False (❌)
    # detail = extra info if something failed
    status = "✅ PASS" if passed else "❌ FAIL"
    results.append({"Rule": rule_name, "Status": status, "Detail": detail})
    print(f"{status} — {rule_name} {detail}")

print("🔧 Validation tracker ready.")

🔧 Validation tracker ready.


In [8]:
# Cell 4 — Rule 1: Row count must be exactly 1470
# We started with 1470 employees.
# Cleaning should never delete real rows.
# If count changed, something went wrong.

expected_rows = 1470
actual_rows = len(df)

check(
    rule_name="Row count = 1470",
    passed=(actual_rows == expected_rows),
    detail=f"(found {actual_rows})"
)

✅ PASS — Row count = 1470 (found 1470)


In [10]:
# Cell 5 — Rule 2: No missing values anywhere
# After cleaning, no cell should be empty.
# A null in Workday = broken employee record.

total_nulls = df.isnull().sum().sum()  # sum of ALL nulls across ALL columns

check(
    rule_name="No null values",
    passed=(total_nulls == 0),
    detail=f"(found {total_nulls} nulls)"
)

✅ PASS — No null values (found 0 nulls)


In [12]:
# Cell 6 — Rule 3: No duplicate worker IDs
# Every employee must have a unique ID.
# Duplicates = two records fighting over same person.

duplicate_ids = df["worker_id"].duplicated().sum()

check(
    rule_name="No duplicate worker_id",
    passed=(duplicate_ids == 0),
    detail=f"(found {duplicate_ids} duplicates)"
)

✅ PASS — No duplicate worker_id (found 0 duplicates)


In [14]:
# Cell 7 — Rule 4: Attrition flag only contains 0 or 1
# We mapped Yes→1, No→0 in Notebook 02.
# Any other value means our mapping broke.

valid_flags = {0, 1}
actual_flags = set(df["employment_attrition_flag"].unique())
flag_ok = actual_flags.issubset(valid_flags)

check(
    rule_name="Attrition flag values are 0 or 1 only",
    passed=flag_ok,
    detail=f"(found values: {actual_flags})"
)

✅ PASS — Attrition flag values are 0 or 1 only (found values: {0, 1})


In [16]:
# Cell 8 — Rule 5: Gender only contains M or F
# We mapped Male→M, Female→F in Notebook 02.
# No other values should exist.

valid_genders = {"M", "F"}
actual_genders = set(df["worker_gender"].unique())
gender_ok = actual_genders.issubset(valid_genders)

check(
    rule_name="Gender values are M or F only",
    passed=gender_ok,
    detail=f"(found values: {actual_genders})"
)

✅ PASS — Gender values are M or F only (found values: {'M', 'F'})


In [18]:
# Cell 9 — Rule 6: Travel frequency only valid codes
# We mapped to NONE, RARE, FREQ in Notebook 02.
# Any leftover original values = mapping failure.

valid_travel = {"NONE", "RARE", "FREQ"}
actual_travel = set(df["travel_frequency"].unique())
travel_ok = actual_travel.issubset(valid_travel)

check(
    rule_name="Travel frequency values valid",
    passed=travel_ok,
    detail=f"(found values: {actual_travel})"
)

✅ PASS — Travel frequency values valid (found values: {'FREQ', 'NONE', 'RARE'})


In [20]:
# Cell 10 — Rule 7: Age must be between 18 and 65
# No employee should be younger than 18 or older than 65.
# These would be data entry errors.

age_ok = df["worker_age"].between(18, 65).all()
bad_ages = df[~df["worker_age"].between(18, 65)]["worker_age"].tolist()

check(
    rule_name="All ages between 18 and 65",
    passed=age_ok,
    detail=f"(bad values: {bad_ages if bad_ages else 'none'})"
)

✅ PASS — All ages between 18 and 65 (bad values: none)


In [22]:
# Cell 11 — Rule 8: Monthly salary must be greater than 0
# No employee can have zero or negative salary.
# That would be corrupt data.

salary_ok = (df["compensation_monthly_salary"] > 0).all()
bad_salaries = df[df["compensation_monthly_salary"] <= 0].shape[0]

check(
    rule_name="All salaries greater than 0",
    passed=salary_ok,
    detail=f"(bad records: {bad_salaries})"
)

✅ PASS — All salaries greater than 0 (bad records: 0)


In [24]:
# Cell 12 — Rule 9: worker_status only Active or Terminated
# We created this column in Notebook 02.
# Only two valid values should exist.

valid_status = {"Active", "Terminated"}
actual_status = set(df["worker_status"].unique())
status_ok = actual_status.issubset(valid_status)

check(
    rule_name="worker_status values valid",
    passed=status_ok,
    detail=f"(found values: {actual_status})"
)

✅ PASS — worker_status values valid (found values: {'Terminated', 'Active'})


In [26]:
# Cell 13 — Rule 10: Column count must be exactly 33
# We started with 35, dropped 3 useless ones = 32,
# then added 1 new column (worker_status) = 33 total.
# Wrong count = something got accidentally dropped or added.

expected_cols = 33
actual_cols = df.shape[1]

check(
    rule_name="Column count = 33",
    passed=(actual_cols == expected_cols),
    detail=f"(found {actual_cols})"
)

✅ PASS — Column count = 33 (found 33)


In [28]:
# Cell 14 — Print the full validation report
# This is your final migration quality report.
# Every rule. Every result. Pass or fail.

print("\n")
print("=" * 55)
print("       WORKDAY MIGRATION VALIDATION REPORT")
print("=" * 55)

passed = sum(1 for r in results if "PASS" in r["Status"])
failed = sum(1 for r in results if "FAIL" in r["Status"])

for r in results:
    print(f"  {r['Status']} | {r['Rule']} {r['Detail']}")

print("=" * 55)
print(f"  TOTAL RULES : {len(results)}")
print(f"  PASSED      : {passed}")
print(f"  FAILED      : {failed}")

if failed == 0:
    print("\n  🎉 ALL CHECKS PASSED. Data is Workday-ready.")
else:
    print(f"\n  ⚠️ {failed} rule(s) failed. Fix before migrating.")

print("=" * 55)



       WORKDAY MIGRATION VALIDATION REPORT
  ✅ PASS | Row count = 1470 (found 1470)
  ✅ PASS | No null values (found 0 nulls)
  ✅ PASS | No duplicate worker_id (found 0 duplicates)
  ✅ PASS | Attrition flag values are 0 or 1 only (found values: {0, 1})
  ✅ PASS | Gender values are M or F only (found values: {'M', 'F'})
  ✅ PASS | Travel frequency values valid (found values: {'FREQ', 'NONE', 'RARE'})
  ✅ PASS | All ages between 18 and 65 (bad values: none)
  ✅ PASS | All salaries greater than 0 (bad records: 0)
  ✅ PASS | worker_status values valid (found values: {'Terminated', 'Active'})
  ✅ PASS | Column count = 33 (found 33)
  TOTAL RULES : 10
  PASSED      : 10
  FAILED      : 0

  🎉 ALL CHECKS PASSED. Data is Workday-ready.


In [30]:
# Cell 15 — Save validation report to file

import os

report_lines = ["WORKDAY MIGRATION VALIDATION REPORT", "=" * 55]
for r in results:
    report_lines.append(f"{r['Status']} | {r['Rule']} {r['Detail']}")
report_lines.append("=" * 55)
report_lines.append(f"PASSED: {passed} / FAILED: {failed}")

report_path = os.path.expanduser("../Data/raw_hr_data.csv")

with open(report_path, "w") as f:
    f.write("\n".join(report_lines))

print(f"💾 Validation report saved!")
print("✅ Notebook 03 complete. Project done.")

💾 Validation report saved!
✅ Notebook 03 complete. Project done.
